In [1]:
import sys 
sys.path.append('../')
import torch
import numpy as np
from scipy.sparse import spdiags
from utils import injection1d, interp2d, gauss_smooth2d
import matplotlib.pyplot as plt
import kornia
import torch.nn.functional as F

In [7]:
k = 5
n = 2**9 + 1

for i in range(5):
    n = (n-1)//2 + 1

In [ ]:
n

In [9]:
np.arange(n)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16])

In [2]:
diags = torch.tensor([[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4]])
offsets = torch.tensor([-1,0,1])
torch.sparse.spdiags(diags, offsets, (4,4)).to_dense()

tensor([[1, 2, 0, 0],
        [1, 2, 3, 0],
        [0, 2, 3, 4],
        [0, 0, 3, 4]])

In [3]:
[513, 1025, 2049, 4097, 8193]

[513, 1025, 2049, 4097, 8193]

In [4]:
n = 8193
nl = 5
[(n-1)//2**(nl-l-1)+1 for l in range(nl)]

[513, 1025, 2049, 4097, 8193]

In [5]:
n = 8193
nl = 5
nc = 3
m = 7
r = 16
x = torch.linspace(-1,1,n)[None,None]
ms = {nl - i:m for i in range(nc)}
kernels = {}

mg_n = []
for l in range(nl):
    mg_n.append(n)
    n = (n-1)//2 + 1
mg_n = mg_n[::-1]
print(mg_n)
for l in range(nl):
    if l == 0:
        kernels['phi-'+str(l)] = torch.nn.Parameter(torch.zeros(1,r,mg_n[l]))
        kernels['psi-'+str(l)] = torch.nn.Parameter(torch.zeros(1,r,mg_n[l]))
    else:
        if l in ms.keys():
            kernels['phi-'+str(l)] = torch.nn.Parameter(torch.zeros(1,r,mg_n[l]))
            kernels['psi-'+str(l)] = torch.nn.Parameter(torch.zeros(1,r,2*ms[l]+1))
kernels = torch.nn.ParameterDict(kernels)

for l in range(nl-1):
    x = injection1d(x)

for l in range(nl):
    if l == 0:
        phi = kernels['phi-'+str(0)]
        psi = kernels['psi-'+str(0)]
        a = torch.einsum('b r n, b r m -> b n m', phi, psi)[None]
        print(a.shape)
        assert a.shape[-1] == x.shape[-1]
    else:
        a = interp2d(a)
        if l in ms.keys():
            phi = torch.rand_like(kernels['phi-'+str(l)])
            psi = torch.rand_like(kernels['psi-'+str(l)])
            a_band = torch.einsum('b r n, b r m -> b m n', phi, psi)[None]
            offsets = torch.arange(-ms[l],ms[l]+1)
            a[0,0] += torch.sparse.spdiags(a_band[0,0], offsets, (mg_n[l],mg_n[l])).to_dense()
            # a += a_band
    a = gauss_smooth2d(a)

[513, 1025, 2049, 4097, 8193]
torch.Size([1, 1, 513, 513])


: 

In [29]:
offsets

tensor([-7, -6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6,  7])